In [1]:
import numpy as np
from pulp import *
import math

## **DEFINE SCENARIO'S FOR EACH STAGE**

##### **ASSUMED YIELD'S FOLLOWS DISCRETE UNIFORM**

In [43]:
possible_t_1 = np.arange(2, 3.1, 0.1)
possible_t_2 = np.arange(2.4, 3.7, 0.1)
possible_t_3 = np.arange(16, 25, 1)


In [44]:
print(f"{possible_t_1}\n{possible_t_2}\n{possible_t_3}")

[2.  2.1 2.2 2.3 2.4 2.5 2.6 2.7 2.8 2.9 3. ]
[2.4 2.5 2.6 2.7 2.8 2.9 3.  3.1 3.2 3.3 3.4 3.5 3.6 3.7]
[16 17 18 19 20 21 22 23 24]


In [45]:
num_scenarios = 3
scenario_stage2 = {s: {"t_1": np.random.choice(possible_t_1),
                       "t_2": np.random.choice(possible_t_2),
                       "t_3" : np.random.choice(possible_t_3)} for s in range(1, num_scenarios + 1)}

scenario_stage3 = {s: {"t_1": np.random.choice(possible_t_1),
                       "t_2": np.random.choice(possible_t_2),
                       "t_3" : np.random.choice(possible_t_3)} for s in range(1, num_scenarios * num_scenarios + 1)}


In [46]:
scenario_stage2

{1: {'t_1': 2.8000000000000007, 't_2': 3.600000000000001, 't_3': 24},
 2: {'t_1': 2.2, 't_2': 3.1000000000000005, 't_3': 16},
 3: {'t_1': 2.900000000000001, 't_2': 2.6, 't_3': 16}}

In [47]:
scenario_stage3

{1: {'t_1': 3.000000000000001, 't_2': 3.700000000000001, 't_3': 19},
 2: {'t_1': 2.3000000000000003, 't_2': 3.1000000000000005, 't_3': 18},
 3: {'t_1': 2.1, 't_2': 2.5, 't_3': 23},
 4: {'t_1': 2.8000000000000007, 't_2': 2.9000000000000004, 't_3': 21},
 5: {'t_1': 2.4000000000000004, 't_2': 2.7, 't_3': 19},
 6: {'t_1': 2.1, 't_2': 2.7, 't_3': 18},
 7: {'t_1': 2.4000000000000004, 't_2': 2.5, 't_3': 20},
 8: {'t_1': 3.000000000000001, 't_2': 3.500000000000001, 't_3': 16},
 9: {'t_1': 2.2, 't_2': 3.600000000000001, 't_3': 21}}

# **NLDS PROBLEMS**

## **NLDS $( t = 1, k = 1 )$**
$ \min \left( 150\ x_{1}^1 + 230\ x_{2}^1 + 260\ x_{3}^1 \right) $

$ subject\ to :  x_{1}^1 +  x_{2}^1 +  x_{3}^1 \leqslant 500 $ 

## **NLDS $( t = 2,\ k = \xi_{i} )\ for\ i \in (1,\ 2,\ 3)$**
$ \min \left( 238\ y_{1}^2 - 170\ w_{1}^2 + 210\ y_{2}^2 - 150\ w_{2}^2 - 36\ w_{3}^2 - 10\ w_{4}^2 + ( 150\ x_{1}^2 + 230\ x_{2}^2 + 260\ x_{3}^2 \right) $

**subject to:**

$ x_{1}^2 +  x_{2}^2 +  x_{3}^2 \leqslant 500$

$ y_{1}^2 -  w_{1}^2 \geqslant 200 - t_{1}.x_{1}^1 $ 

$ y_{2}^2 -  w_{2}^2 \geqslant 240 - t_{2}.x_{2}^1 $

$ w_{3}^2 + w_{4}^2 	\leqslant t_{3}.x_{3}^1$ , $ w_{3}^2 	\leqslant 6000$

$ x_{1}^2 +  x_{2}^2 +  x_{3}^2 \leqslant 500 $ 

**where** $ \xi = (t_{1},\ t_{2},\ t_{3})$

## **NLDS $( t = 3, k = \xi_{i}) for\ i \in (1,\ 2,\ ...,\ 9) $**
$ \min \left( 238\ y_{1}^3 - 170\ w_{1}^3 + 210\ y_{2}^3 - 150\ w_{2}^3 - 36\ w_{3}^3 - 10\ w_{4}^3 \right) $

**subject to:**

$ y_{1}^3 -  w_{1}^3 \geqslant 200 - t_{1}.x_{1}^2 $ 

$ y_{2}^3 -  w_{2}^3 \geqslant 240 - t_{2}.x_{2}^2 $

$ w_{3}^3 + w_{4}^3 \leqslant t_{3}.x_{3}^2$ , $ w_{3}^3 \leqslant 6000$

**where** $ \xi = (t_{1},\ t_{2},\ t_{3})$

## **FOR EACH NLDS ($t, k$) A TUPLE ($t, k$) IS DEFINED**

In [48]:
nodes= {
    1 : (1, 1),
    2 : (2, 1), 3 : (2, 2), 4 : (2, 3),
    5 : (3, 1), 6 : (3, 2), 7 : (3, 3),
    8 : (3, 4), 9 : (3, 5), 10 : (3, 6),
    11 : (3, 7), 12 : (3, 8), 13 : (3, 9)
}

## **DEFINE A MAP OF SCENARIOS FOR IDENTIFYING PARENT AND CHILD NODES**

In [49]:
child_to_parent_map = {
    (2, 1): (1, 1), (2, 2): (1, 1), (2, 3): (1, 1),
    (3, 1): (2, 1), (3, 2): (2, 1), (3, 3): (2, 1),
    (3, 4): (2, 2), (3, 5): (2, 2), (3, 6): (2, 2),
    (3, 7): (2, 3), (3, 8): (2, 3), (3, 9): (2, 3),
}

parent_to_children_map = {
    (1, 1): [(2, 1), (2, 2), (2, 3)],
    (2, 1): [(3, 1), (3, 2), (3, 3)],
    (2, 2): [(3, 4), (3, 5), (3, 6)],
    (2, 3): [(3, 7), (3, 8), (3, 9)],
}

In [50]:
numeric_solutions = {}
duals = {}
dual = list()

node = (1, 1)
x_1 = LpVariable.dicts(f'x_{node}', range(1, 4), lowBound=0)
theta_1 = LpVariable(f'theta_{node}', lowBound=-1e9)
model = LpProblem(f"numeric_S1_N1", LpMinimize)
model += 150*x_1[1] + 230*x_1[2] + 260*x_1[3] + theta_1
model += x_1[1] + x_1[2] + x_1[3] <= 500
model.solve()
numeric_solutions[node] = [x_1[s].varValue for s in range(1, 4)]

for name, c in model.constraints.items():
    dual.append(c.pi)
duals[node] = dual
    

for s_idx in range(1, 4):
    node = (2, s_idx)
    parent_node = (1, 1)
    dual = list()
    x_1_parent = numeric_solutions[parent_node]
    params = scenario_stage2[s_idx]
    
    x_2 = LpVariable.dicts(f'x_{node}', range(1, 4), lowBound=0)
    y_2 = LpVariable.dicts(f'y_{node}', range(1, 3), lowBound=0)
    w_2 = LpVariable.dicts(f'w_{node}', range(1, 5), lowBound=0)
    theta_2 = LpVariable(f'theta_{node}', lowBound=-1e9)

    
    model = LpProblem(f"numeric_S2_N{s_idx}", LpMinimize)
    model += (238*y_2[1] - 170*w_2[1] + 210*y_2[2] - 150*w_2[2] - 36*w_2[3] - 10*w_2[4] +
              150*x_2[1] + 230*x_2[2] + 260*x_2[3] + theta_2)
    model += x_2[1] + x_2[2] + x_2[3] <= 500
    model += y_2[1] - w_2[1] >= 200 - params["t_1"] * x_1_parent[0]
    model += y_2[2] - w_2[2] >= 240 - params["t_2"] * x_1_parent[1]
    model += w_2[3] + w_2[4] <= params["t_3"] * x_1_parent[2]
    model += w_2[3] <= 6000
    model.solve()
    for name, c in model.constraints.items():
        dual.append(c.pi)
    duals[node] = dual
    numeric_solutions[node] = ([x_2[s].varValue for s in range(1, 4)], [y_2[s].varValue for s in range(1, 3)], [w_2[s].varValue for s in range(1, 5)])
    

for s_idx in range(1, 10):
    node = (3, s_idx)
    parent_node = child_to_parent_map[node] 
    x_2_parent = numeric_solutions[parent_node][0]
    params = scenario_stage3[s_idx]
    dual = list()

    y_3 = LpVariable.dicts(f'y_{node}', range(1, 3), lowBound=0)
    w_3 = LpVariable.dicts(f'w_{node}', range(1, 5), lowBound=0)

    
    model = LpProblem(f"numeric_S3_N{s_idx}", LpMinimize)
    model += (238*y_3[1] - 170*w_3[1] + 210*y_3[2] - 150*w_3[2] - 36*w_3[3] - 10*w_3[4])
    model += y_3[1] - w_3[1] >= 200 - params["t_1"] * x_2_parent[0]
    model += y_3[2] - w_3[2] >= 240 - params["t_2"] * x_2_parent[1]
    model += w_3[3] + w_3[4] <= params["t_3"] * x_2_parent[2]
    model += w_3[3] <= 6000
    model.solve()
    for name, c in model.constraints.items():
        dual.append(c.pi)
    duals[node] = dual
    numeric_solutions[node] = ([y_3[s].varValue for s in range(1, 3)], [w_3[s].varValue for s in range(1, 5)])
    



##### **EACH NLDS SOLVED (FOR FIRST FORWARD PASS)**

In [51]:
numeric_solutions

{(1, 1): [0.0, 0.0, 0.0],
 (2, 1): ([0.0, 0.0, 0.0], [200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (2, 2): ([0.0, 0.0, 0.0], [200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (2, 3): ([0.0, 0.0, 0.0], [200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 1): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 2): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 3): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 4): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 5): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 6): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 7): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 8): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0]),
 (3, 9): ([200.0, 240.0], [0.0, 0.0, 0.0, 0.0])}

In [52]:
duals

{(1, 1): [0.0],
 (2, 1): [0.0, 238.0, 210.0, -36.0, 0.0],
 (2, 2): [0.0, 238.0, 210.0, -36.0, 0.0],
 (2, 3): [0.0, 238.0, 210.0, -36.0, 0.0],
 (3, 1): [238.0, 210.0, -36.0, 0.0],
 (3, 2): [238.0, 210.0, -36.0, 0.0],
 (3, 3): [238.0, 210.0, -36.0, 0.0],
 (3, 4): [238.0, 210.0, -36.0, 0.0],
 (3, 5): [238.0, 210.0, -36.0, 0.0],
 (3, 6): [238.0, 210.0, -36.0, 0.0],
 (3, 7): [238.0, 210.0, -36.0, 0.0],
 (3, 8): [238.0, 210.0, -36.0, 0.0],
 (3, 9): [238.0, 210.0, -36.0, 0.0]}

# **DEFINE A DICTIONARY FOR SYMBOLIC MODELS**

#### **its necessary to have symbolic models for extracting coefficent matrixes**

In [53]:
symbolic_models = {}
symbolic_vars = {}

# Stage 1
node = (1, 1)
var_name_base = f"s{node[0]}_n{node[1]}"
x_1 = LpVariable.dicts(f'x_{var_name_base}', range(1, 4), lowBound=0)
theta_1 = LpVariable(f'theta_{var_name_base}', lowBound=-1e9)
symbolic_vars[node] = {'x': x_1, 'theta': theta_1}
model = LpProblem(f"Symbolic_{var_name_base}", LpMinimize)
model += 150*x_1[1] + 230*x_1[2] + 260*x_1[3] + theta_1
model += x_1[1] + x_1[2] + x_1[3] <= 500
symbolic_models[node] = model

# Stage 2
for s_idx in range(1, 4):
    node = (2, s_idx)
    parent_node = (1, 1)
    

    var_name_base = f"s{node[0]}_n{node[1]}"
    parent_var_name_base = f"s{parent_node[0]}_n{parent_node[1]}"
    
    x_1_parent = symbolic_vars[parent_node]['x']
    params = scenario_stage2[s_idx]
    
    x_2 = LpVariable.dicts(f'x_{var_name_base}', range(1, 4), lowBound=0)
    y_2 = LpVariable.dicts(f'y_{var_name_base}', range(1, 3), lowBound=0)
    w_2 = LpVariable.dicts(f'w_{var_name_base}', range(1, 5), lowBound=0)
    theta_2 = LpVariable(f'theta_{var_name_base}', lowBound=-1e9)
    symbolic_vars[node] = {'x': x_2, 'y': y_2, 'w': w_2, 'theta': theta_2}
    
    model = LpProblem(f"Symbolic_{var_name_base}", LpMinimize)
    model += (238*y_2[1] - 170*w_2[1] + 210*y_2[2] - 150*w_2[2] - 36*w_2[3] - 10*w_2[4] +
              150*x_2[1] + 230*x_2[2] + 260*x_2[3] + theta_2)
    model += x_2[1] + x_2[2] + x_2[3] <= 500
    model += y_2[1] - w_2[1] >= 200 - params["t_1"] * x_1_parent[1]
    model += y_2[2] - w_2[2] >= 240 - params["t_2"] * x_1_parent[2]
    model += -w_2[3] - w_2[4] >= -params["t_3"] * x_1_parent[3]
    model += w_2[3] <= 6000
    symbolic_models[node] = model

# Stage 3
for s_idx in range(1, 10):
    node = (3, s_idx)
    parent_node = child_to_parent_map[node]
    

    var_name_base = f"s{node[0]}_n{node[1]}"
    parent_var_name_base = f"s{parent_node[0]}_n{parent_node[1]}"
    
    x_2_parent = symbolic_vars[parent_node]['x']
    params = scenario_stage3[s_idx]

    y_3 = LpVariable.dicts(f'y_{var_name_base}', range(1, 3), lowBound=0)
    w_3 = LpVariable.dicts(f'w_{var_name_base}', range(1, 5), lowBound=0)
    symbolic_vars[node] = {'y': y_3, 'w': w_3}
    
    model = LpProblem(f"Symbolic_{var_name_base}", LpMinimize)
    model += (238*y_3[1] - 170*w_3[1] + 210*y_3[2] - 150*w_3[2] - 36*w_3[3] - 10*w_3[4])
    model += y_3[1] - w_3[1] >= 200 - params["t_1"] * x_2_parent[1]
    model += y_3[2] - w_3[2] >= 240 - params["t_2"] * x_2_parent[2]
    model += -w_3[3] - w_3[4] >= -params["t_3"] * x_2_parent[3]
    model += w_3[3] <= 6000
    symbolic_models[node] = model

In [54]:
symbolic_models

{(1,
  1): Symbolic_s1_n1:
 MINIMIZE
 1*theta_s1_n1 + 150*x_s1_n1_1 + 230*x_s1_n1_2 + 260*x_s1_n1_3 + 0
 SUBJECT TO
 _C1: x_s1_n1_1 + x_s1_n1_2 + x_s1_n1_3 <= 500
 
 VARIABLES
 -1000000000 <= theta_s1_n1 Continuous
 x_s1_n1_1 Continuous
 x_s1_n1_2 Continuous
 x_s1_n1_3 Continuous,
 (2,
  1): Symbolic_s2_n1:
 MINIMIZE
 1*theta_s2_n1 + -170*w_s2_n1_1 + -150*w_s2_n1_2 + -36*w_s2_n1_3 + -10*w_s2_n1_4 + 150*x_s2_n1_1 + 230*x_s2_n1_2 + 260*x_s2_n1_3 + 238*y_s2_n1_1 + 210*y_s2_n1_2 + 0
 SUBJECT TO
 _C1: x_s2_n1_1 + x_s2_n1_2 + x_s2_n1_3 <= 500
 
 _C2: - w_s2_n1_1 + 2.8 x_s1_n1_1 + y_s2_n1_1 >= 200
 
 _C3: - w_s2_n1_2 + 3.6 x_s1_n1_2 + y_s2_n1_2 >= 240
 
 _C4: - w_s2_n1_3 - w_s2_n1_4 + 24 x_s1_n1_3 >= 0
 
 _C5: w_s2_n1_3 <= 6000
 
 VARIABLES
 -1000000000 <= theta_s2_n1 Continuous
 w_s2_n1_1 Continuous
 w_s2_n1_2 Continuous
 w_s2_n1_3 Continuous
 w_s2_n1_4 Continuous
 x_s1_n1_1 Continuous
 x_s1_n1_2 Continuous
 x_s1_n1_3 Continuous
 x_s2_n1_1 Continuous
 x_s2_n1_2 Continuous
 x_s2_n1_3 Continuo

## **DEFINE A FUNCTION FOR EXTRACTING COEFFICENT MATRIX'S**

In [55]:
def get_coeffs(prob, parent_vars_list, current_stage_x_vars_list):

    constraints = prob.constraints
    
    T = list()
    W = list() 
    h = list()
    
    current_vars_list = [v for v in prob.variables() if v not in parent_vars_list]

    for name, constraint in constraints.items():
        h.append(-constraint.constant)
        T_row = [constraint.get(x, 0) for x in parent_vars_list]
        T.append(T_row)
        W_row = [constraint.get(y, 0) for y in current_vars_list]
        W.append(W_row)
        
    return {"W": W}, {"T": T}, {"h": h}

### **w_prime creator**

In [63]:
def build_w_prime(prob):
    """a function that generates feasibility_check problem from given problem"""

    constraints = list(prob.constraints.values())
    n_constraints = len(constraints)
    w_prime = LpProblem("feasibility-check-problem", LpMinimize)
    v = LpVariable.dicts("v", ((i, j) for i in range(1, n_constraints + 1) for j in (1, 2)), lowBound=0, cat="Continuous")
    w_prime += lpSum(v[i, j] for i in range(1, n_constraints + 1) for j in (1, 2))
    for index, c in enumerate(constraints, start=1):
        expr = c + v[index, 1] - v[index, 2]
        w_prime += expr
    return w_prime, v

# **GENERATE FEASIBILITY CUTS**

In [64]:

def generate_feasibility_cut(infeasible_node, duals_sigma, symbolic_models, parent_symbolic_vars):

    symbolic_subproblem = symbolic_models[infeasible_node]

    # Extract coefficient matrices from the symbolic model
    _, T, h = get_coeffs(symbolic_subproblem, list(parent_symbolic_vars['x'].values()), [])

    T_matrix = np.array(T['T'])
    h_vector = np.array(h['h'])
    sigma_vector = np.array(duals_sigma)

    D_vector = np.dot(sigma_vector, T_matrix)
    d_scalar = np.dot(sigma_vector, h_vector)

    # Return the cut data in a dictionary
    cut_info = {
        'type': 'feasibility',
        'D_coeffs': D_vector,
        'd_rhs': d_scalar
    }

    parent_x_vars = parent_symbolic_vars['x']
    cut_expression_str = " + ".join([f"{D_vector[j]:.2f} * {parent_x_vars[j+1].name}" for j in range(len(D_vector))]) + f" >= {d_scalar:.2f}"
    print(f"  Generated Feasibility Cut: {cut_expression_str}")

    return cut_info

# **solve subproblem**

In [57]:
def solve_subproblem(node, parent_solution, existing_cuts):

    stage, scenario_index = node
    var_name_base = f"s{stage}_n{scenario_index}"
    model_name = f"NLDS_{var_name_base}"
    model = LpProblem(model_name, LpMinimize)
    
    local_x = None
    local_theta = None

    if stage == 1:
        x_1 = LpVariable.dicts(f'x_{var_name_base}', range(1, 4), lowBound=0)
        theta_1 = LpVariable(f'theta_{var_name_base}', lowBound=-1e9)
        local_x, local_theta = x_1, theta_1
        
        model += 150*x_1[1] + 230*x_1[2] + 260*x_1[3] + theta_1, "Objective_S1"
        model += x_1[1] + x_1[2] + x_1[3] <= 500, "Capacity_S1"

    elif stage == 2:
        params = scenario_stage2[scenario_index]
        t1, t2, t3 = params["t_1"], params["t_2"], params["t_3"]
        
        x_2 = LpVariable.dicts(f'x_{var_name_base}', range(1, 4), lowBound=0)
        y_2 = LpVariable.dicts(f'y_{var_name_base}', range(1, 3), lowBound=0)
        w_2 = LpVariable.dicts(f'w_{var_name_base}', range(1, 5), lowBound=0)
        theta_2 = LpVariable(f'theta_{var_name_base}', lowBound=-1e9)
        local_x, local_theta = x_2, theta_2
        
        model += (238*y_2[1] - 170*w_2[1] + 210*y_2[2] - 150*w_2[2] - 36*w_2[3] - 10*w_2[4] +
                  150*x_2[1] + 230*x_2[2] + 260*x_2[3] + theta_2), f"Objective_{var_name_base}"
        
        x1_vals = parent_solution['x_values']
        model += x_2[1] + x_2[2] + x_2[3] <= 500, f"Capacity_{var_name_base}"
        model += y_2[1] - w_2[1] >= 200 - t1 * x1_vals[1], f"Link1_{var_name_base}"
        model += y_2[2] - w_2[2] >= 240 - t2 * x1_vals[2], f"Link2_{var_name_base}"
        model += -w_2[3] - w_2[4] >= -t3 * x1_vals[3], f"Link3_{var_name_base}"
        model += w_2[3] <= 6000, f"W3_Bound_{var_name_base}"

    elif stage == 3:
        params = scenario_stage3[scenario_index]
        t1, t2, t3 = params["t_1"], params["t_2"], params["t_3"]
        
        y_3 = LpVariable.dicts(f'y_{var_name_base}', range(1, 3), lowBound=0)
        w_3 = LpVariable.dicts(f'w_{var_name_base}', range(1, 5), lowBound=0)
        
        
        model += (238*y_3[1] - 170*w_3[1] + 210*y_3[2] - 150*w_3[2] - 36*w_3[3] - 10*w_3[4]), f"Objective_{var_name_base}"
        
        x2_vals = parent_solution['x_values']
        model += y_3[1] - w_3[1] >= 200 - t1 * x2_vals[1], f"Link1_{var_name_base}"
        model += y_3[2] - w_3[2] >= 240 - t2 * x2_vals[2], f"Link2_{var_name_base}"
        model += -w_3[3] - w_3[4] >= -t3 * x2_vals[3], f"Link3_{var_name_base}"
        model += w_3[3] <= 6000, f"W3_Bound_{var_name_base}"


    if existing_cuts:
        for i, cut_info in enumerate(existing_cuts):
            cut_type = cut_info['type']
            cut_name = f"{cut_type}_cut_{var_name_base}_{i}"
            
            if cut_type == 'optimality':
                E_coeffs = cut_info['E_coeffs']
                e_rhs = cut_info['e_rhs']
                cut_expr = local_theta + lpSum(E_coeffs[j] * local_x[j+1] for j in range(len(E_coeffs))) >= e_rhs
                model.addConstraint(cut_expr, name=cut_name)
            
            elif cut_type == 'feasibility':
                D_coeffs = cut_info['D_coeffs']
                d_rhs = cut_info['d_rhs']
                cut_expr = lpSum(D_coeffs[j] * local_x[j+1] for j in range(len(D_coeffs))) >= d_rhs
                model.addConstraint(cut_expr, name=cut_name)

    model.solve(PULP_CBC_CMD(msg=False))
    
    if model.status == LpStatusOptimal:
        solution_vars = {v.name: v.varValue for v in model.variables()}
        
        x_prefix = f"x_{var_name_base}"
        theta_name = f"theta_{var_name_base}"
        
        x_vars_dict = {
            int(name.rsplit('_', 1)[-1]): val 
            for name, val in solution_vars.items() 
            if name.startswith(x_prefix)
        }

        theta_val = solution_vars.get(theta_name, 0)
        
        solution_dict = {
            'objective': value(model.objective),
            'x_values': x_vars_dict,
            'theta': theta_val,
            'all_vars': solution_vars
        }
        
        pi_duals, rho_duals, sigma_duals = {}, {}, {}
        for name, c in model.constraints.items():
            rhs = -c.constant
            if 'optimality_cut' in name:
                rho_duals[name] = {'pi': c.pi, 'rhs': rhs}
            elif 'feasibility_cut' in name:
                sigma_duals[name] = {'pi': c.pi, 'rhs': rhs}
            else:
                pi_duals[name] = {'pi': c.pi, 'rhs': rhs}

        duals_dict = {'pi': pi_duals, 'rho': rho_duals, 'sigma': sigma_duals}
        
        return 'feasible', {'solution': solution_dict, 'duals': duals_dict}
        
    else: # Model is Infeasible
        
        
        feasibility_prob, _ = build_w_prime(model)
        feasibility_prob.solve(PULP_CBC_CMD(msg=False))
        
        # Check if the feasibility problem has a non-zero objective, indicating true infeasibility
        if value(feasibility_prob.objective) > 1e-6:
            duals_sigma = [const.pi for const in feasibility_prob.constraints.values()]
            return 'infeasible', {'solution': None, 'duals': {'sigma': duals_sigma}}
        else:
            return 'infeasible', {'solution': None, 'duals': None}

# **ADD CUT TO NODE**

In [58]:
def add_cut_to_node(node, cut, all_cuts):

    if node not in all_cuts:
        all_cuts[node] = []
    all_cuts[node].append(cut)

# **GENERATE OPTIMALITY CUT**

In [59]:
PROB_S3_GIVEN_S2 = 1.0 / 3.0

def generate_optimality_cut(parent_node, children_nodes, solutions, duals, symbolic_models, parent_symbolic_vars):
    """
    Generates an optimality cut by returning its coefficients and RHS value.
    """
    parent_solution = solutions[parent_node]
    parent_x_values = np.array(list(parent_solution['x_values'].values()))

    parent_theta_var = parent_symbolic_vars['theta']
    parent_x_vars = parent_symbolic_vars['x']

    num_parent_vars = len(parent_x_vars)
    expected_E = np.zeros(num_parent_vars)
    expected_e = 0.0

    for child_node in children_nodes:
        child_duals = duals.get(child_node)
        if not child_duals:
            continue

        e_contribution = 0.0
        
        # Get symbolic model and variables to extract coefficients
        child_symbolic_model = symbolic_models[child_node]
        childs_parent_vars = list(parent_symbolic_vars['x'].values())
        child_x_vars_dict = symbolic_vars[child_node].get('x', {})
        child_x_vars = list(child_x_vars_dict.values())
        
        _, T, h = get_coeffs(child_symbolic_model, childs_parent_vars, child_x_vars)
        
        h_vector = np.array(h['h'])
        pi_k = np.array([d['pi'] for d in child_duals['pi'].values()])
        e_contribution += np.dot(pi_k, h_vector)
        
        if child_duals.get('rho'):
            rho_k = np.array([d['pi'] for d in child_duals['rho'].values()])
            d_k = np.array([d['rhs'] for d in child_duals['rho'].values()])
            e_contribution += np.dot(rho_k, d_k)
            
        if child_duals.get('sigma'):
            sigma_k = np.array([d['pi'] for d in child_duals['sigma'].values()])
            f_k = np.array([d['rhs'] for d in child_duals['sigma'].values()])
            e_contribution += np.dot(sigma_k, f_k)


        T_matrix = np.array(T['T'])
        E_contribution = np.dot(pi_k, T_matrix)
        
        expected_e += PROB_S3_GIVEN_S2 * e_contribution
        expected_E += PROB_S3_GIVEN_S2 * E_contribution

    actual_future_cost = expected_e - np.dot(expected_E, parent_x_values)
    estimated_future_cost = parent_solution['theta']

    if actual_future_cost > estimated_future_cost + 1e-6:
        
        cut_info = {
            'type': 'optimality',
            'E_coeffs': expected_E,
            'e_rhs': expected_e
        }
        
        cut_expression_str = f"{parent_theta_var.name} + " + \
                             " + ".join([f"{expected_E[j]:.2f} * {parent_x_vars[j+1].name}" for j in range(num_parent_vars)]) + \
                             f" >= {expected_e:.2f}"

        print(f"  Generated Optimality Cut: {cut_expression_str} for {parent_node}\n")
        return cut_info
    
    else:
        return None

# **HELPER FUNCTIONS**

In [60]:
def calculate_expected_future_cost(parent_node, children_nodes, solutions):
    expected_cost = 0.0

    conditional_prob = 1.0 / len(children_nodes)
    
    for child_node in children_nodes:

        child_objective_value = solutions[child_node]['objective']
        expected_cost += conditional_prob * child_objective_value
        
    return expected_cost

In [61]:
def get_symbolic_vars_for_node(node, all_symbolic_vars):
    return all_symbolic_vars.get(node)

## **MAIN CODE**

In [73]:
MAX_STAGES = 3
iteration_count = 0
is_optimal = False


cuts_added_in_backward_pass = False

current_stage = 1
current_scenario_index = 1
direction = "FORE"

solutions = {} 
duals = {}     
cuts = {}      

while not is_optimal:
    
    current_node = (current_stage, current_scenario_index)

    if direction == "FORE":

        if current_node == (1, 1):
            iteration_count += 1
            print(f"\n{'='*20} STARTING ITERATION {iteration_count} {'='*20}")
            print("--- FORWARD PASS ---")
            

            cuts_added_in_backward_pass = False
        
        print(f"Solving Node: {current_node}")

        parent_node = child_to_parent_map.get(current_node)
        parent_solution = solutions.get(parent_node)
        existing_cuts = cuts.get(current_node, [])

        status, result = solve_subproblem(current_node, parent_solution, existing_cuts)

        if status == "infeasible":
            if result['duals'] and result['duals'].get('sigma'):
                duals_sigma = result['duals']['sigma']
                parent_vars = get_symbolic_vars_for_node(parent_node, symbolic_vars)
                
                feasibility_cut_info = generate_feasibility_cut(current_node, duals_sigma, symbolic_models, parent_vars)
            
                if feasibility_cut_info is not None:
                    add_cut_to_node(parent_node, feasibility_cut_info, cuts)
            
            current_stage, current_scenario_index = parent_node
            
        else: 
            print(f"                       Feasible.\n")
            solutions[current_node] = result['solution']
            duals[current_node] = result['duals']


            if current_node == (1, 1):
                sol = result['solution']
                x_vals_str = ", ".join([f"x{i}: {val:.2f}" for i, val in sorted(sol['x_values'].items())])
                print(f">>> Stage 1 Results for Iteration {iteration_count}:")
                print(f"    Decision Variables: [ {x_vals_str} ]")
                print(f"    Objective (Cost + Theta): {sol['objective']:.2f}\n")
            
            if current_stage == 1:
                current_stage = 2
                current_scenario_index = 1
            elif current_stage == 2:
                if current_scenario_index < 3:
                    current_scenario_index += 1
                else:
                    current_stage = 3
                    current_scenario_index = 1
            elif current_stage == 3:
                if current_scenario_index < 9:
                    current_scenario_index += 1
                else:
                    print("\n--- BACKWARD PASS ---")
                    direction = "BACK"
                    current_stage = 2 

    elif direction == "BACK":
        print(f"Generating optimality cuts for Stage {current_stage}")
        any_cut_added_in_this_stage = False


        if current_stage == 1:
            nodes_in_stage = [(1, 1)]
        else:
            nodes_in_stage = [node for node in parent_to_children_map if node[0] == current_stage]
        
        for parent_node in nodes_in_stage:
            if parent_node == (1, 1):
  
                children_nodes = [node for node in solutions if node[0] == 2]
            else:
                children_nodes = parent_to_children_map[parent_node]
            
            actual_future_cost = calculate_expected_future_cost(parent_node, children_nodes, solutions)
            estimated_future_cost = solutions[parent_node]['theta']

            if actual_future_cost > estimated_future_cost + 1e-6:
                print(f"  -> Optimality gap for {parent_node}. Generating cut.")
                any_cut_added_in_this_stage = True
                
                parent_vars = get_symbolic_vars_for_node(parent_node, symbolic_vars)
                optimality_cut_info = generate_optimality_cut(parent_node, children_nodes, solutions, duals, symbolic_models, parent_vars)

                if optimality_cut_info is not None:
                    add_cut_to_node(parent_node, optimality_cut_info, cuts)

        if any_cut_added_in_this_stage:
            cuts_added_in_backward_pass = True


        if current_stage == 1:
 
            if not any_cut_added_in_this_stage:
                is_optimal = True
                print("\nConvergence reached: No new cut added to Stage 1.")
            else:
     
                direction = "FORE"
                current_stage = 1
                current_scenario_index = 1
        else:

            current_stage -= 1 

if is_optimal:
    print("\n--- FINAL OPTIMAL SOLUTION ---\n")
    final_decision = solutions[(1, 1)]
    
    # first stage
    x1_values = [val for val in final_decision['x_values'].values()]
    print(f"First stage decision (x_1): {x1_values}")
    print(f"Total expected cost: {final_decision['objective']:}")

    # second stage
    print("\n--- Second Stage Recourse Decisions ---")

    for s in range(1, 4):
        node = (2, s)
        if node in solutions:
            stage_2_solution = solutions[node]
            x2_values = stage_2_solution.get('x_values', {})

            x_vals_str = ", ".join([f"x{i}: {val:}" for i, val in sorted(x2_values.items())])
            
            print(f"  For Scenario {s} (Node {node}):")
            print(f"    Optimal second-stage decision (x_2): [ {x_vals_str} ]")


==================== STARTING ITERATION 1 ====================
--- FORWARD PASS ---
Solving Node: (1, 1)
                       Feasible.

>>> Stage 1 Results for Iteration 1:
    Decision Variables: [ x1: 0.00, x2: 0.00, x3: 0.00 ]
    Objective (Cost + Theta): -1000000000.00

Solving Node: (2, 1)
                       Feasible.

Solving Node: (2, 2)
                       Feasible.

Solving Node: (2, 3)
                       Feasible.

Solving Node: (3, 1)
                       Feasible.

Solving Node: (3, 2)
                       Feasible.

Solving Node: (3, 3)
                       Feasible.

Solving Node: (3, 4)
                       Feasible.

Solving Node: (3, 5)
                       Feasible.

Solving Node: (3, 6)
                       Feasible.

Solving Node: (3, 7)
                       Feasible.

Solving Node: (3, 8)
                       Feasible.

Solving Node: (3, 9)
                       Feasible.


--- BACKWARD PASS ---
Generating optimality cuts for Stage 